In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [19]:
from datasets import load_dataset


stories = load_dataset("roneneldan/TinyStories")


In [21]:
raw_texts = stories['train']['text'][:50000]

In [22]:
def preprocess(text):
    return text.replace('\n', ' ').strip()

texts = [preprocess(t) for t in raw_texts]

In [13]:
def text_to_bytes(text):
    return list(text.encode('utf-8'))

In [ ]:
class Tokenizer:
    def __init__(self):
        self.vocab = dict()
        self.word2idx = dict()
        self.idx2word = dict()

    def upload_texts(self, texts: list[str]):
        words = set()
        for text in texts:
            words.update(text.split())
        
        # Add special tokens to vocab
        special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]
        self.vocab = {word: i for i, word in enumerate(special_tokens + sorted(words), start=0)}
        self.word2idx = self.vocab
        self.idx2word = {i: word for word, i in self.vocab.items()}

    def upload_vocab(self, vocab: dict[str, int]): 
        self.vocab = vocab
        self.word2idx = vocab
        self.idx2word = {i: word for word, i in self.vocab.items()}

    def encode(self, text: str) -> list[int]:
        # dict.get(key, default_value)
        # self.word2idx.get(w, self.word2idx['<unk>'])
        return [self.word2idx.get(w, self.word2idx.get("<unk>", 0)) for w in text.split()]

    def decode(self, ids: list[int]) -> str:
        # self.idx2word.get(i, "<unk>")
        return " ".join([self.idx2word[i] for i in ids])
    
    def get_vocab_size(self) -> int:
        return len(self.vocab)

# remove non alphabetical character
import re
import json
def clean_text(txt):
    txt = re.sub(r"[^a-zA-Z0-9.,!?'\s]", "", txt)
    txt = re.sub(r"\s+", " ", txt)
    return txt.strip().lower()

In [35]:
tokenizer = Tokenizer()

tokenizer.upload_texts(texts)

In [46]:
sample_text = "one day a little dog named"
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: one day a little dog named
Encoded: [37567, 21502, 12199, 33448, 22642, 36286]
Decoded: one day a little dog named


In [47]:
import json
json.dump(tokenizer.vocab, open("tiny_stories_vocab.json", "w"))